<a href="https://colab.research.google.com/github/anyuanay/INFO212/blob/main/INFO212_Week7_Lecture2_Aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

## Week 7: Lecture 2: Data Aggregation and Group Operations
---

**Agenda:**
- frame.groupby(list).agg
- groupby by a function
- groupby by a dictionary
- groupby by a list
- iterate over groups
- split-apply-combine
- pivotal tables and cross-tabulation


In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# DataSet: Meals and Tips
Download the `tips.csv` dataset and load it as a DataFrame here.

## Explore the tips dataset

## Add a tip_pct column
Compute the percentage of tips for each meal

# Grouping with Dicts and Series

for the default axis=0. A dict or a function passed to groupby is applied to the index labels, one per row.

- A string (groupby("key")) refers to a column, and its values become the keys.
- A dict or function is matched against the index labels.
- A Series or array aligned with the rows is used directly as the keys.
- With axis=1, the dict or function is applied to the column labels instead.

To group by a function or dict applied to column values, first map the column with .map() and pass the result.

## Grouping information may exist in a form other than an array. Let’s consider another example DataFrame:

```
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people
```

## Now, suppose I have a group correspondence for the columns and want to sum together the columns by group:

```
mapping = {'a': 'red', 'b': 'red', 'c': 'blue',
           'd': 'blue', 'e': 'red', 'f' : 'orange'}
```

## Now, you could construct an array from this dict to pass to groupby, but instead we can just pass the dict:

```
by_column = people.groupby(mapping, axis=1)
by_column.sum()
```

## Exercise:
For the day column, map 'Sun' and 'Sat' to 'weekend' and 'Thur' and 'Fri' to weekday. Compute the mean total_bill and tip_pct of weekend and weekday.

# Grouping with Functions
Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. Any function passed as a group key will be called once per index
value, with the return values being used as the group names. More concretely, consider
the example DataFrame above, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the len
function:

```
for k, chunk in people.groupby(len):
    print(k)
    print(chunk)
```

## Exercise:
Define a function that converts the tip_pct to percentage (100%) and round the percentage to the nearest number multiplied by 10 (i.e, 10, 20, 30, ...). For each rounded percentage, compute the average total_bill and tip_pct.

# Grouping by Index Levels
## A convenience for hierarchically indexed datasets is the ability to aggregate using one of the levels of an axis index. Let’s look at an example:

```
columns = pd.MultiIndex.from_arrays([['US', 'US', 'US', 'JP', 'JP'],
                                    [1, 3, 5, 1, 3]],
                                    names=['cty', 'tenor'])
hier_df = pd.DataFrame(np.random.randn(4, 5), columns=columns)
hier_df
```

## To group by level, pass the level number or name using the level keyword:

```
hier_df.groupby(level='cty', axis=1).count()
```

# Retrieval Practice

# Data Aggregation
Aggregations refer to any data transformation that produces scalar values from
arrays. The preceding examples have used several of them, including mean, count,
min, and sum. However, you are not limited to only this set of
methods.

```
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})
df
```

You can use aggregations of your own devising and additionally call any method that
is also defined on the grouped object. For example, you might recall that quantile
computes sample quantiles of a Series or a DataFrame’s columns.
While quantile is not explicitly implemented for GroupBy, it is a Series method and
thus available for use. Internally, GroupBy efficiently slices up the Series, calls
piece.quantile(0.9) for each piece, and then assembles those results together into
the result object:

```
grouped = df.groupby('key1')
grouped['data1'].quantile(0.9)
```

## To use your own aggregation functions, pass any function that aggregates an array to the aggregate or agg method:

```
def peak_to_peak(arr):
    return arr.max() - arr.min()

df[['data1', 'data2']].apply(peak_to_peak)

df.groupby('key1').agg(peak_to_peak)
```

## You may notice that some methods like describe also work, even though they are not aggregations, strictly speaking:

```
df.describe()

grouped.describe()
```

## Exercise:
For each unique value in the day column, compute the maximum, mininum, and the difference between the max and min for total_bill and tip_pct.

## Select Top Results on Aggregation
Suppose we wanted to select the top five tip_pct values by group. First, write a function that selects the rows with the largest values in a particular column:

```
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column, ascending=False)[:n]

```

Now, if we group by smoker, say, and call apply with this function, we get the following:

```
tips.groupby('smoker').apply(top)
```

The top function is called on each row group from the
DataFrame, and then the results are glued together using pandas.concat, labeling the
pieces with the group names. The result therefore has a hierarchical index whose
inner level contains index values from the original DataFrame.

## Exercise:
For each unique value in the day column, extract the top 3 total bills.

In the preceding examples, you see that the resulting object has a hierarchical index formed from the group keys along with the indexes of each piece of the original object. You can disable this by passing group_keys=False to groupby:

```
tips.groupby('smoker', group_keys=False).apply(top)
```

# What is a Pivot Table and How to Use It?
- A pivot table is a data summarization tool frequently found in spreadsheet programs
and other data analysis software.
- It aggregates a table of data by one or more keys,
arranging the data in a rectangle with some of the group keys along the rows and
some along the columns.
- Pivot tables in Python with pandas are made possible
through the groupby facility combined with reshape operations
utilizing hierarchical indexing.
- DataFrame has a pivot_table method, and
there is also a top-level pandas.pivot_table function.
- In addition to providing a
convenience interface to groupby, pivot_table can add partial totals, also known as
margins.

## Returning to the `tips` dataset, suppose you wanted to compute a table of group means (the default pivot_table aggregation type) arranged by day and smoker on the rows:
```
tips[['day', 'smoker', 'total_bill', 'tip']].pivot_table(index=['day', 'smoker']).mean()
```

## This could have been produced with groupby directly. Now, suppose we want to aggregate only tip_pct and size, and additionally group by time. I’ll put smoker in the table columns and day in the rows:
```
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker')
```

## We could augment this table to include partial totals by passing margins=True. This has the effect of adding All row and column labels, with corresponding values being the group statistics for all the data within a single tier:
```
tips.pivot_table(['size', 'tip_pct'], index=['time', 'day'],
                 columns='smoker', margins=True)
```

## Here, the All values are means without taking into account smoker versus nonsmoker (the All columns) or any of the two levels of grouping on the rows (the All row).

## To use a different aggregation function, pass it to aggfunc. For example, 'count' or len will give you a cross-tabulation (count or frequency) of group sizes:
```
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',
                 aggfunc=len, margins=True)
```

## If some combinations are empty (or otherwise NA), you may wish to pass a fill_value:
```
tips.pivot_table('tip_pct', index=['time', 'size', 'smoker'],
                 columns='day', aggfunc='mean', fill_value=0)
```

# What is Cross-Tabulation? How to Use Crosstab?
## A cross-tabulation (or crosstab for short) is a special case of a pivot table that computes group frequencies. Here is an example:

```
import pandas as pd
from io import StringIO
data = """\
Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""
data = pd.read_table(StringIO(data), sep='\s+')
```

## As part of some survey analysis, we might want to summarize this data by nationality and handedness. You could use pivot_table to do this, but the pandas.crosstab function can be more convenient:

```
pd.crosstab(data.Nationality, data.Handedness, margins=True)
```

## The first two arguments to crosstab can each either be an array or Series or a list of arrays. As in the tips data:

```
pd.crosstab([tips.time, tips.day], tips.smoker, margins=True).plot.bar()
```

# Retrieval Practice